# 00 · Preflight: Scientific Python for Geodesy

*Build exactly the array, plotting, and convention skills used by the course.*

**Learning objectives**

- predict NumPy shapes produced by indexing and broadcasting
- distinguish elementwise multiplication, matrix multiplication, and solves
- create a labeled matplotlib figure with a colorbar
- convert units and coordinates into GeoDef's public conventions
- use seeds, validation messages, and tolerant floating-point comparisons

**Prerequisites:** none  
**Estimated time:** 60–90 minutes; skippable after the self-test


## Self-test

1. What shape results from `(4, 1) + (3,)`?
2. Does `a[:2]` usually copy data or view it?
3. What does a colorbar label need in addition to a quantity name?
4. Convert 12 km to GeoDef's base length unit.
5. Is public depth positive up or down?

<details><summary>Answers and skip guidance</summary>

The answers are `(4, 3)`, a view, physical units, `12_000 m`, and positive
down. If those answers and `@` versus `*` are familiar, continue directly to
Chapter 01.

</details>


## Arrays are shaped scientific statements

A one-dimensional `(N,)` array means one value per station or patch. `(N, 3)`
usually means one row per item with three named coordinates/components. A
length-`2N` slip vector contains two blocked components, not `N` three-vectors.
Slicing usually returns a view; integer-array and boolean indexing return
copies. Shapes are part of the interface, so inspect them before interpreting
values.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

values = np.arange(8.0)
view = values[:4]
mask_copy = values[values % 2 == 0]
column = np.arange(4.0)[:, np.newaxis]
row = np.array([10.0, 20.0, 30.0])
print("view/copy shapes:", view.shape, mask_copy.shape)
print("broadcast shape:", (column + row).shape)


## Broadcasting, vectorization, and linear algebra

Broadcasting aligns trailing dimensions; dimensions match when equal or when
one is 1. `np.newaxis` makes the intended row/column role visible. Vectorized
expressions state the operation over all stations or patches and let optimized
array kernels do the work.

`*` is elementwise. `@` contracts matrix dimensions. Prefer `solve(A, b)` to
`inv(A) @ b`, and `lstsq(G, d)` when the system is rectangular. Factorizations
are more stable than forming explicit inverses.


In [ ]:
G_demo = np.array([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])
d_demo = np.array([1.0, 2.1, 2.9])
model_demo, *_ = np.linalg.lstsq(G_demo, d_demo, rcond=None)
assert np.allclose(G_demo @ model_demo, [1.05, 2.0, 2.95])
print("least-squares intercept and slope:", model_demo)


## Plot anatomy and honest labels

A figure contains one or more axes; each plotted quantity needs a title or
legend, axis labels, and units. A colorbar is another measurement axis and must
name its quantity and unit. Shared limits are essential for visual comparison.


In [ ]:
x = np.linspace(0.0, 10.0, 25)
y = np.sin(x)
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
points = ax.scatter(x, y, c=y, cmap="RdBu_r", vmin=-1.0, vmax=1.0)
ax.set(title="A fully labeled diagnostic", xlabel="Distance (km)", ylabel="Amplitude (m)")
fig.colorbar(points, ax=ax, label="Amplitude (m)");


## Units, coordinates, roundoff, and reproducibility

GeoDef uses meters for lengths/slip, degrees for angles, longitude then
latitude in named geographic APIs, ENU local axes, and depth positive down.
Every local array belongs to a `LocalFrame`. Read the complete
[conventions](../docs/conventions.md) before importing external data.

Binary floating point rounds most real numbers. Compare computed arrays with
`np.allclose`, using a tolerance justified by scale, rather than exact `==`.
Create randomness with `np.random.default_rng(seed)` and record the seed.
Validation errors are field-specific evidence; `.validate()` adds interactive
warnings without changing the object.


In [ ]:
good_fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=8_000.0, strike=90.0, dip=30.0,
    length=20_000.0, width=10_000.0,
)
print(good_fault.validate())
assert np.allclose(0.1 + 0.2, 0.3)


## Closing preview: the complete workflow

The same code is annotated in the [five-minute quickstart](../docs/quickstart.md).
Chapter 01 explains `Fault` and displacement; 02 explains the hidden matrix;
03–04 explain `solve` and regularization; 07 explains bounds; named prediction
plots return in Chapters 03 and 05.


In [ ]:
rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=8_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=6, n_width=3,
)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 7), np.linspace(33.88, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
centers = fault.centers_local
true_dip_slip = 1.2 * np.exp(
    -(centers[:, 0] / 7_000.0) ** 2
    - ((centers[:, 1] + 2_000.0) / 5_000.0) ** 2
)
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_dip_slip
)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, 0.004, east.size),
    north=north + rng.normal(0.0, 0.004, north.size),
    up=up + rng.normal(0.0, 0.008, up.size),
    sigma_east=0.004, sigma_north=0.004, sigma_up=0.008,
    name="synthetic_gnss",
)
result = geodef.solve(
    fault, datasets=gnss, components="dip",
    regularization="laplacian", regularization_strength=1.0,
    bounds=(0.0, None),
)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
geodef.plot.slip(
    fault, result.dip_slip, ax=axes[0], title="Recovered dip slip",
    colorbar_label="Slip (m)",
)
geodef.plot.prediction(result, ax=axes[1]);


## Checkpoint questions

**Why is `(N, 1)` useful in broadcasting?**
<details><summary>Answer</summary>It makes each of N values a row so a `(K,)`
array broadcasts across K columns.</details>

**Why does `np.linalg.solve` beat an explicit inverse?**
<details><summary>Answer</summary>It uses a factorization without constructing a
less stable, more expensive inverse matrix.</details>

**What makes a random experiment reproducible?**
<details><summary>Answer</summary>A recorded generator algorithm/API and explicit
seed, plus fixed inputs and software environment.</details>


## Common mistakes

- `reshape` can preserve size while silently changing scientific meaning.
- A plotted color without units is not interpretable data.
- Mixing `lat, lon` tables with named `lon, lat` APIs can move a model globally.
- Exact floating-point equality fails for harmless roundoff and can pass for
  values that are exactly equal but scientifically wrong.


## Recap

- Shapes encode which axis represents stations, patches, or components.
- Broadcasting and `@` replace loops when the mathematics is vectorized.
- Every diagnostic plot carries labels, units, and comparable limits.
- GeoDef errors and validation reports protect physical conventions.

Use the [workflow map](../docs/workflow.md) to choose your next API level, then
continue to Chapter 01.


## Exercises

Worked solutions are in `solutions/00_preflight_solutions.ipynb`.

1. Pack fake strike/dip arrays into `[strike | dip]`, then unpack them.
2. Vectorize a loop that evaluates `a * x + b` for four slopes and 20 x values.
3. Convert `[lat, lon, depth_km]` rows to `[lon, lat, depth_m]`.
4. Fix calls with negative depth, dip above 90°, and mismatched station arrays.
5. Challenge: take an unlabeled two-panel plot and make comparison defensible.


## Further reading

- Harris et al. (2020), NumPy array programming.
- Hunter (2007), matplotlib's figure/axes design.
- Higham (2002), floating-point accuracy and stability.
- [GeoDef conventions](../docs/conventions.md), the binding project reference.
